In [ ]:
import pandas as pd
import numpy as np
import scipy
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn
import torch    
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import regex as re  
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

In [ ]:
df_test = pd.read_csv('datasets/banking77_test.csv')
df_train = pd.read_csv('datasets/banking77_train.csv')

print(df_test.head())
print(df_train.head())

In [ ]:
pd.isnull(df_test).sum(
)
pd.isnull(df_train).sum(
)

print(df_test.duplicated().sum())
print(df_train.duplicated().sum())

In [ ]:
df_test.dtypes
df_train.dtypes

In [ ]:
df_test['category'] = df_test['category'].astype('category')
df_train['category'] = df_train['category'].astype('category')

In [ ]:
df_test.describe(include='all')
df_train.describe(include='all')

In [ ]:
print("Label distribution in the training set:")
print(df_train['category'].value_counts())
label_pcts = df_train['category'].value_counts(normalize=True) * 100
print(label_pcts)

In [ ]:
df_train['Query length'] = df_train['text'].astype(str).apply(lambda x: len(x.split()))
length_stats = df_train.groupby('category')['Query length'].describe().reset_index()
print(length_stats)

In [ ]:
df_test['Query length'] = df_test['text'].astype(str).apply(lambda x: len(x.split()))

length_stats = df_test.groupby('category')['Query length'].describe().reset_index()
print(length_stats)

In [ ]:
# Find most commonly used words in the training set

def split_into_words(text):
    lowercase_text = text.lower()
    split_words = re.split("\W+", lowercase_text)
    return split_words

number_of_words = df_train['text'].astype(str).apply(lambda x: len(x.split()))  

stopwords = ['a', 'are', 'am', 'an', 'the', 'my', 'you', 'it', 'and', 'to', 'me', 'has', 'had', 'will', 'was', 'on', 'of', 'be', 'from', 'was', 'have', 'in', 'for', 'is', 'can', 'do', 'i', 'or', 'but', 'if', 'then', 'else', 'when', 'where', 'why', 'how', 'what', 'which', 'who', 'whom', 'whose', 'this', 'that', 'these', 'those']

full_text = df_train['text'].str.lower().str.cat(sep=' ')

clean_text = re.sub(r'[!"\$%&\'()*+,\-.\/:;=#@?\[\\\]^_`{|}~]', ' ', full_text)

meaningful_words = [word for word in clean_text.split() if word not in stopwords]
meaningful_words_count = Counter(meaningful_words)
most_frequent_meaningful_words = meaningful_words_count.most_common(30)

print(most_frequent_meaningful_words)


In [ ]:
# TF-IDF Vectorisation

label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(df_train['category'])
y_test = label_encoder.transform(df_test['category'])

# Save number of unique classes for MLP output layer
num_classes = len(label_encoder.classes_)
print(f"Total target intent classes: {num_classes}")


tfidf_vectoriser = TfidfVectorizer(
    stop_words='english', 
    ngram_range=(1,2), 
    max_features=5000, 
    token_pattern=r'\b\w\w+\b')
X_train_tfidf = tfidf_vectoriser.fit_transform(df_train['text'])
X_test_tfidf = tfidf_vectoriser.transform(df_test['text'])

print(f"X_train shape: {X_train_tfidf.shape}")
print(f"X_test shape: {X_test_tfidf.shape}")    



In [ ]:
# Print 10 random features captured by vectoriser
import random
feature_names = tfidf_vectoriser.get_feature_names_out()
print("Sample extracted features (including bigrams):")
print(random.sample(list(feature_names), 10))

In [ ]:
# Prepare data arrays
# Convert sparse TF-IDF matrices to dense arrays for PyTorch

X_train_dense = X_train_tfidf.toarray().astype(np.float32)
X_test_dense = X_test_tfidf.toarray().astype(np.float32)

# Custom PyTorch Dataset to handle input features and label integer indices
class BankingIntentDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long) # CrossEntropy expects labels as long integers

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

# Instatitate datasets and dataloaders
train_dataset = BankingIntentDataset(X_train_dense, y_train)
test_dataset = BankingIntentDataset(X_test_dense, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
# Build and train MLP architecture
class IntentMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(IntentMLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

input_dim = X_train_dense.shape[1]
hidden_dim = 128
output_dim = num_classes

model = IntentMLP(input_dim, hidden_dim, output_dim)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimiser = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

In [ ]:
epochs = 10
print("Starting training")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for batch_x, batch_y in train_loader:
        optimiser.zero_grad() # Clear gradient tracking buffers

        # Forward pass
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)

        # Backward pass and parameter updates
        loss.backward()
        optimiser.step()

        # Track training metrics
        running_loss += loss.item() * batch_x.size(0)
        _, predicted = torch.max(outputs, 1)
        correct_predictions += (predicted == batch_y).sum().item()
        total_samples += batch_y.size(0)
    
    # Calculate performance metrics per epoch
    epoch_loss = running_loss / total_samples
    epoch_accuracy = (correct_predictions / total_samples) * 100
    print(f"Epoch {epoch+1:02d}/{epochs}, Loss average: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.2f}%")


In [ ]:
# Evaluate on test dataset
model.eval()
all_predictions = []
all_actuals = []

print("Evaluating on test dataset")

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        outputs = model(batch_x)
        _, predicted = torch.max(outputs, 1)

        all_predictions.extend(predicted.numpy())
        all_actuals.extend(batch_y.numpy())

print(f"Overall test accuracy: {accuracy_score(all_actuals, all_predictions) * 100:.2f}%")
print(f"F1 score: {f1_score(all_actuals, all_predictions, average='macro'):.4f}")
print(f"Weighted F1 score: {f1_score(all_actuals, all_predictions, average='weighted'):.4f}")

print(classification_report(all_actuals, all_predictions, target_names=label_encoder.classes_))


In [ ]:
# Finetuning BERT transformer

tokeniser = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokeniser(
    list(df_train['text'].astype(str)),
    truncation=True,
    padding=True,
    max_length=64, # EDA confirmed max length of 53 words so reducing max length for efficiency
    return_tensors='pt'
)

test_encodings = tokeniser(
    list(df_test['text'].astype(str)),
    truncation=True,
    padding=True,
    max_length=64,
    return_tensors='pt'
)

print(f"Train input_ids shape: {train_encodings['input_ids'].shape}") 

In [ ]:
# Transformer dataset and dataloaders
class TransformerIntentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Extract items for this specific row index
        item = {key: val[idx] for key, val in self.encodings.items()}
        # Add target integer label to the dictionary 
        item['labels'] = self.labels[idx]
        return item

# Instantiate training and testing datasets
transformer_train_dataset = TransformerIntentDataset(train_encodings, y_train)
transformer_test_dataset = TransformerIntentDataset(test_encodings, y_test)

# Setup dataloaders for handling model batches
transformer_train_dataloader = DataLoader(transformer_train_dataset, batch_size=32, shuffle=True)
transformer_test_dataloader = DataLoader(transformer_test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Initialise and fine-tune BERT

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using {device} for training")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"Using {device} for training")
else:
    device = torch.device("cpu")
    print(f"Using {device} for training")

In [ ]:
# Load pre-trained BERT architecture with custom classification
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_classes)
model.to(device)

In [ ]:
# Instantiate BERT model for sequence classification
optimiser = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

# Core fine-tuning loop
epochs = 3
print("Starting BERT fine-tuning")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    progress_bar = tqdm(transformer_train_dataloader, desc=f"Epoch {epoch+1}/{epochs}")

    for batch in progress_bar:
        optimiser.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        # Backward pass and parameter tuning
        loss.backward()
        optimiser.step()

        # Track training metrics    
        running_loss += loss.item() * input_ids.size(0)
        _, predicted = torch.max(logits, 1)
        correct_predictions += (predicted == labels).sum().item()
        total_samples += labels.size(0)

        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})    
            
    epoch_loss = running_loss / total_samples
    epoch_accuracy = (correct_predictions / total_samples) * 100
    print(f"Epoch {epoch+1:02d}/{epochs}, Loss average: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.2f}%")


    

In [ ]:
# Evaluate on test dataset
model.eval()
all_transformer_predictions = []
all_transformer_actuals = []

print("Evaluating BERT predictions")

with torch.no_grad():
    for batch in tqdm(transformer_test_dataloader, desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device) 
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
    
        _, predicted = torch.max(logits, 1) # Identify class with highest probability

        all_transformer_predictions.extend(predicted.cpu().numpy())
        all_transformer_actuals.extend(labels.cpu().numpy())


print(f"Overall test accuracy: {accuracy_score(all_transformer_actuals, all_transformer_predictions) * 100:.2f}%")
print(f"F1 score: {f1_score(all_transformer_actuals, all_transformer_predictions, average='macro'):.4f}")
print(f"Weighted F1 score: {f1_score(all_transformer_actuals, all_transformer_predictions, average='weighted'):.4f}")

print(classification_report(all_transformer_actuals, all_transformer_predictions, target_names=label_encoder.classes_))


In [ ]:
# Visualise the comparison of each performance output
metrics = ['Accuracy', 'Macro F1 score', 'Weighted F1 score']

mlp_scores = [0.8495, 0.8501, 0.8501]
bert_scores = [0.8916, 0.8873, 0.8873]

x = np.arange(len(metrics))  
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

mlp_colour = '#6fa8dc'
bert_colour = '#8fce00'

rects1 = ax.bar(x - width/2, mlp_scores, width, label='Baseline MLP(TF-IDF)', color=mlp_colour, edgecolor='white')
rects2 = ax.bar(x + width/2, bert_scores, width, label='Fined-Tuned BERT', color=bert_colour, edgecolor='white')

# Set y-axis limit slightly above 1 to make space for labels
ax.set_ylim(0, 1.2)

# Add grid lines for easier reading
ax.grid(axis='y', linestyle='--', alpha=0.5)

# Function to add numerical labels on top of the bars
def add_labels(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.4f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 5),  # 5 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

# Run the label function for both bar groups
add_labels(rects1)
add_labels(rects2)

ax.set_title('Performance Comparison: MLP vs BERT Intent Classification', fontsize=16, fontweight='bold', pad=20)
ax.set_ylabel('Score (0.0 - 1.0)', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.legend(fontsize=11, loc='lower right')
